In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("data.csv")
df.shape
X = df.drop(["id", "diagnosis"], axis=1)
y = df["diagnosis"]

print(X.shape, y.shape)

y.replace("M", 1, inplace=True)
y.replace("B", 0, inplace=True)


(569, 31) (569,)


0      1
1      1
2      1
3      1
4      1
      ..
564    1
565    1
566    1
567    1
568    0
Name: diagnosis, Length: 569, dtype: object

In [10]:
prior_1 = y.sum() / len(y)
prior_0 = 1 - prior_1

In [11]:
X_1 = X[y==1]
X_0 = X[y==0]

In [12]:
means_1 = {}
variances_1 = {}
means_0 = {}
variances_0 = {}

for col_name in X_1.columns:
    means_1[col_name] = X_1[col_name].mean()
    variances_1[col_name] = X_1[col_name].var()

for col_name in X_0.columns:
    means_0[col_name] = X_0[col_name].mean()
    variances_0[col_name] = X_0[col_name].var()

In [13]:
def calculate_probability(x, mean, variance):
    return (1 / np.sqrt(2 * np.pi * variance)) * np.exp(-np.square(x - mean) / (2 * variance))

In [14]:
def naive_bayes(x):
    score_1 = prior_1
    for col in X_1.columns:
        score_1 *= calculate_probability(x[col], means_1[col], variances_1[col])
    score_0 = prior_0
    for col in X_1.columns:
        score_0 *= calculate_probability(x[col], means_0[col], variances_0[col])

    if score_1 > score_0:
        return 1
    else:
        return 0

In [15]:
predictions = []

for index, x in X.iterrows():
    predictions.append(naive_bayes(x))

In [16]:
from sklearn.metrics import classification_report

classification_report(y.astype(int), predictions)

/home/jathin/.conda/envs/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jathin/.conda/envs/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jathin/.conda/envs/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

'              precision    recall  f1-score   support\n\n           0       0.63      1.00      0.77       357\n           1       0.00      0.00      0.00       212\n\n    accuracy                           0.63       569\n   macro avg       0.31      0.50      0.39       569\nweighted avg       0.39      0.63      0.48       569\n'